# Persian MMS/VITS fine-tuning on Kaggle

This notebook fine-tunes Meta's `facebook/mms-tts-fas` checkpoint on Persian speech. Before running it, enable **GPU** and **Internet** in Kaggle's Notebook options. The default configuration uses 1,000 ManaTTS samples for a practical first run.

> License note: Meta MMS weights are CC BY-NC 4.0, so the resulting weights are for non-commercial use.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/ylacombe/finetune-hf-vits.git'
REPO_DIR = Path('/kaggle/working/finetune-hf-vits')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', 'checkout', '6f3f51f4d667f5c3eef89484d151ffd39d2c2b89'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print('Python:', sys.version)

In [ ]:
# Install versions known to work with the upstream training recipe.
%pip install -q transformers==4.35.2 datasets[audio]==2.16.1 accelerate==0.25.0 huggingface-hub==0.20.3 numpy==1.26.4 pyarrow==14.0.2 pandas==2.1.4 librosa==0.10.1 soundfile==0.12.1 matplotlib==3.8.2 tensorboard==2.15.1 Cython==3.0.8

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Kaggle, open Notebook options and select a GPU accelerator.')
print('GPU:', torch.cuda.get_device_name(0))

# Apply the two small Kaggle adaptations to the pinned upstream revision.
script_path = REPO_DIR / 'run_vits_finetuning.py'
source = script_path.read_text(encoding='utf-8')
if 'dataset_data_files: Optional[str]' not in source:
    source = source.replace(
        '    overwrite_cache: bool = field(',
        "    dataset_data_files: Optional[str] = field(default=None, metadata={'help': 'Optional path or URL passed as data_files.'})\n    overwrite_cache: bool = field(",
        1,
    )
    source = source.replace(
        '    raw_datasets = DatasetDict()\n',
        "    raw_datasets = DatasetDict()\n    dataset_kwargs = {'data_files': data_args.dataset_data_files} if data_args.dataset_data_files else {}\n",
        1,
    )
    source = source.replace(
        '            token=model_args.token,\n        )',
        '            token=model_args.token,\n            **dataset_kwargs,\n        )',
    )
    source = source.replace(
        '        model.save_pretrained(training_args.output_dir)\n',
        '        model.save_pretrained(training_args.output_dir)\n        feature_extractor.save_pretrained(training_args.output_dir)\n        tokenizer.save_pretrained(training_args.output_dir)\n',
        1,
    )
    script_path.write_text(source, encoding='utf-8')

(REPO_DIR / 'monotonic_align' / 'monotonic_align').mkdir(exist_ok=True)
subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    cwd=REPO_DIR / 'monotonic_align',
    check=True,
)

## Training settings

Keep the defaults for the first end-to-end run. For a longer run, increase `TRAIN_SAMPLES` and `EPOCHS`. The first 25 records are reserved for evaluation.

In [ ]:
import json

TRAIN_SAMPLES = 1000
EPOCHS = 20
BATCH_SIZE = 8
OUTPUT_DIR = Path('/kaggle/working/persian-mms-tts')
BASE_TRAIN_DIR = Path('/kaggle/working/mms-tts-fas-train')

config = {
    'project_name': 'persian_mms_vits_kaggle',
    'push_to_hub': False,
    'overwrite_output_dir': False,
    'output_dir': str(OUTPUT_DIR),
    'report_to': ['tensorboard'],
    'dataset_name': 'parquet',
    'dataset_data_files': 'https://huggingface.co/datasets/MahtaFetrat/Mana-TTS/resolve/main/dataset/dataset_part_001.parquet',
    'audio_column_name': 'audio',
    'text_column_name': 'transcript',
    'train_split_name': f'train[25:{25 + TRAIN_SAMPLES}]',
    'eval_split_name': 'train[:25]',
    'full_generation_sample_text': 'سلام، این یک نمونه از مدل تبدیل متن فارسی به گفتار است.',
    'max_duration_in_seconds': 15,
    'min_duration_in_seconds': 1.0,
    'max_tokens_length': 350,
    'model_name_or_path': str(BASE_TRAIN_DIR),
    'preprocessing_num_workers': 2,
    'do_train': True,
    'num_train_epochs': EPOCHS,
    'gradient_accumulation_steps': 2,
    'gradient_checkpointing': False,
    'per_device_train_batch_size': BATCH_SIZE,
    'learning_rate': 2e-5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.99,
    'warmup_ratio': 0.01,
    'group_by_length': False,
    'do_eval': True,
    'eval_steps': 100,
    'per_device_eval_batch_size': BATCH_SIZE,
    'max_eval_samples': 25,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3,
    'weight_fmaps': 1,
    'weight_gen': 1,
    'weight_kl': 1.5,
    'weight_duration': 1,
    'weight_mel': 35,
    'logging_steps': 10,
    'save_steps': 250,
    'save_total_limit': 2,
    'fp16': True,
    'seed': 42,
}
runtime_config = Path('/kaggle/working/finetune_mms_fa_runtime.json')
runtime_config.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(runtime_config.read_text(encoding='utf-8'))

## Prepare the Persian training checkpoint

The public inference checkpoint contains the generator. Fine-tuning also needs the original discriminator, so the upstream conversion utility downloads and combines both once.

In [ ]:
if not (BASE_TRAIN_DIR / 'config.json').exists():
    subprocess.run(
        [
            sys.executable,
            'convert_original_discriminator_checkpoint.py',
            '--language_code', 'fas',
            '--pytorch_dump_folder_path', str(BASE_TRAIN_DIR),
        ],
        cwd=REPO_DIR,
        check=True,
    )
print('Training checkpoint:', BASE_TRAIN_DIR)

## Fine-tune

Training checkpoints and TensorBoard logs are written under `/kaggle/working/persian-mms-tts`. Rerunning this cell resumes from the latest checkpoint when the output directory is preserved.

In [ ]:
env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run(
    [
        'accelerate', 'launch',
        '--num_processes', '1',
        '--mixed_precision', 'fp16',
        'run_vits_finetuning.py', str(runtime_config),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

## Test the fine-tuned voice

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer, VitsModel

text = 'سلام، این یک نمونه از صدای فارسی آموزش‌دیده روی کگل است.'
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = VitsModel.from_pretrained(OUTPUT_DIR).to('cuda').eval()
inputs = tokenizer(text, return_tensors='pt').to('cuda')
with torch.no_grad():
    waveform = model(**inputs).waveform[0].cpu().numpy()

wav_path = Path('/kaggle/working/persian-test.wav')
sf.write(wav_path, waveform, model.config.sampling_rate)
display(Audio(filename=str(wav_path)))
print('Saved:', wav_path)

In [ ]:
import shutil

archive = shutil.make_archive('/kaggle/working/persian-mms-tts', 'zip', OUTPUT_DIR)
print('Download this file from the Kaggle Output panel:', archive)